# 🎙️ Personal AI Voice Studio - Colab GPU Worker (Coqui XTTS-v2)

This Google Colab notebook provides **free cloud GPU inference** for your Personal AI Voiceover & Voice Cloning Platform.

### Instructions:
1. **Set Runtime to GPU**: Go to `Runtime` -> `Change runtime type` -> Select **T4 GPU** (or A100/V100).
2. **Set your Backend URL**: Set `BACKEND_URL` to your hosted app URL (e.g., `https://your-app.run.app`).
3. **Run All Cells**: Click `Runtime` -> `Run all`.
4. The worker will register with your backend and process text-to-speech & voice cloning jobs automatically!

In [ ]:
# ==========================================
# WORKER CONFIGURATION
# ==========================================
import os

# Replace this with your AI Studio / Cloud Run deployment URL or tunnel address
BACKEND_URL = os.environ.get("BACKEND_URL", "https://ais-dev-nfcdemywbugjmmfkwn4n6j-211529956336.europe-west2.run.app")
WORKER_API_KEY = os.environ.get("WORKER_API_KEY", "voice_studio_secret_worker_key_2026")
WORKER_ID = "colab_gpu_worker_t4"

print(f"📌 Target Backend: {BACKEND_URL}")
print(f"🔑 Worker API Key: {WORKER_API_KEY[:4]}***")

In [ ]:
# ==========================================
# 1. GPU & SYSTEM DETECTION
# ==========================================
import torch
import psutil

if not torch.cuda.is_available():
    raise RuntimeError("❌ GPU is NOT available in this Colab runtime. Go to Runtime -> Change runtime type -> GPU.")

device_name = torch.cuda.get_device_name(0)
vram_total_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)
vram_free_mb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / (1024 ** 2)
cuda_version = torch.version.cuda
ram_mb = psutil.virtual_memory().total / (1024 ** 2)

print(f"✅ GPU Detected: {device_name}")
print(f"📊 Total VRAM: {vram_total_mb:.0f} MB | Free VRAM: {vram_free_mb:.0f} MB")
print(f"⚙️ CUDA Version: {cuda_version}")
print(f"💻 System RAM: {ram_mb:.0f} MB")

In [ ]:
# ==========================================
# 2. DEPENDENCY INSTALLATIONS
# ==========================================
!pip install -q TTS requests pydub ffmpeg-python psutil
!apt-get update -qq && !apt-get install -y -qq ffmpeg

print("✅ All worker dependencies installed.")

In [ ]:
# ==========================================
# 3. LOAD COQUI XTTS-v2 MODEL
# ==========================================
import time
from TTS.api import TTS

print("⏳ Loading Coqui XTTS-v2 model onto GPU...")

In [ ]:
# Agree to Coqui XTTS Terms automatically for non-commercial/personal use
os.environ["COQUI_TOS_AGREED"] = "1"

start_time = time.time()
try:
    tts = TTS(model_name="tts_models/multilingual/multi-dataset/xtts_v2").to("cuda")
    print(f"🎉 Coqui XTTS-v2 loaded successfully onto GPU in {time.time() - start_time:.1f}s!")
except Exception as e:
    print(f"❌ Model load failed: {e}")
    raise e

In [ ]:
# ==========================================
# 4. WORKER EVENT LOOP & JOB POLLING
# ==========================================
import requests
import tempfile
import wave
import contextlib

headers = {
    "X-Worker-Api-Key": WORKER_API_KEY,
    "Content-Type": "application/json"
}

def register_worker():
    payload = {
        "worker_id": WORKER_ID,
        "gpu_name": device_name,
        "vram_total_mb": int(vram_total_mb),
        "vram_free_mb": int(vram_free_mb),
        "cuda_version": str(cuda_version),
        "ram_total_mb": int(ram_mb),
        "xtts_loaded": True
    }
    try:
        res = requests.post(f"{BACKEND_URL}/api/worker/register", json=payload, headers=headers, timeout=10)
        if res.status_code == 200:
            print(f"🟢 Worker '{WORKER_ID}' registered with backend.")
        else:
            print(f"⚠️ Register warning ({res.status_code}): {res.text}")
    except Exception as e:
        print(f"❌ Connection error registering worker: {e}")

def send_heartbeat():
    payload = {
        "worker_id": WORKER_ID,
        "vram_free_mb": int((torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / (1024 ** 2)),
        "xtts_loaded": True,
        "status": "online"
    }
    try:
        requests.post(f"{BACKEND_URL}/api/worker/heartbeat", json=payload, headers=headers, timeout=5)
    except Exception:
        pass

def process_job(job):
    job_id = job["job_id"]
    text = job["text"]
    language = job.get("language", "en")
    voice_profile = job.get("voice_profile", {})
    ref_audio_url = voice_profile.get("reference_audio_url")
    
    print(f"\n⚡ Processing Job [{job_id}] | Language: {language} | Text length: {len(text)} chars")
    
    # 1. Obtain Reference Audio File
    ref_wav_path = None
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_ref:
        ref_wav_path = tmp_ref.name
    
    try:
        if ref_audio_url:
            print(f"📥 Downloading reference audio from {ref_audio_url}...")
            r = requests.get(ref_audio_url, timeout=30)
            with open(ref_wav_path, "wb") as f:
                f.write(r.content)
        else:
            # Create fallback 3-sec reference tone if no reference provided
            import numpy as np
            from scipy.io import wavfile
            sr = 24000
            t = np.linspace(0, 3, int(sr * 3))
            sine = (np.sin(2 * np.pi * 220 * t) * 16384).astype(np.int16)
            wavfile.write(ref_wav_path, sr, sine)

        # 2. Run Coqui XTTS-v2 Speech Generation
        out_wav_path = f"/tmp/out_{job_id}.wav"
        print(f"🤖 Running XTTS-v2 GPU inference...")
        
        tts.tts_to_file(
            text=text,
            speaker_wav=ref_wav_path,
            language=language,
            file_path=out_wav_path
        )
        
        # Get audio duration
        duration = 5.0
        try:
            with contextlib.closing(wave.open(out_wav_path, 'r')) as f:
                frames = f.getnframes()
                rate = f.getframerate()
                duration = frames / float(rate)
        except Exception:
            pass

        # 3. Post result back to backend
        print(f"📤 Uploading result for job [{job_id}] ({duration:.1f}s audio)...")
        with open(out_wav_path, "rb") as audio_f:
            files = {"audio_file": (f"{job_id}.wav", audio_f, "audio/wav")}
            data = {
                "status": "completed",
                "duration": str(duration),
                "sample_rate": "24000",
                "format": "wav"
            }
            res = requests.post(
                f"{BACKEND_URL}/api/worker/jobs/{job_id}/result",
                headers={"X-Worker-Api-Key": WORKER_API_KEY},
                data=data,
                files=files,
                timeout=60
            )
            if res.status_code == 200:
                print(f"✅ Job [{job_id}] completed successfully!")
            else:
                print(f"❌ Failed uploading result ({res.status_code}): {res.text}")
                
    except Exception as e:
        print(f"❌ Failure executing job [{job_id}]: {e}")
        # Report error back to backend
        try:
            requests.post(
                f"{BACKEND_URL}/api/worker/jobs/{job_id}/result",
                headers={"X-Worker-Api-Key": WORKER_API_KEY},
                json={"status": "failed", "error": str(e)},
                timeout=10
            )
        except Exception:
            pass
    finally:
        if ref_wav_path and os.path.exists(ref_wav_path):
            try: os.unlink(ref_wav_path)
            except Exception: pass

# Start event loop
register_worker()
print("🚀 Colab GPU Worker is active and listening for generation jobs...")

last_hb = 0
while True:
    try:
        now = time.time()
        if now - last_hb > 15:
            send_heartbeat()
            last_hb = now
            
        # Poll for pending job
        res = requests.get(f"{BACKEND_URL}/api/worker/jobs", headers=headers, timeout=5)
        if res.status_code == 200:
            job = res.json()
            process_job(job)
        else:
            time.sleep(2.5)
    except KeyboardInterrupt:
        print("\n🛑 Worker stopped by user.")
        break
    except Exception as e:
        time.sleep(3)